<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/dentistry/lecture_3/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%96_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа № 3: Обработка естественного языка в клинической медицине (стоматологии)**

## Введение

В лекции №3 мы познакомились с ключевыми этапами обработки естественного языка (NLP) применительно к медицинским текстам. Мы разобрали, как «живой», неструктурированный и зашумленный язык (жалобы пациентов, записи в амбулаторной карте, рентгенологические заключения, описания микробиологических исследований) превращается в структурированные числовые признаки, пригодные для машинного обучения и статистического анализа. Теперь вам предстоит применить эти знания на практике.

Цель данной работы — закрепить навыки:

- очистки текста (токенизация, удаление стоп-слов, нормализация, лемматизация),
- векторизации текста методами Bag of Words и TF‑IDF,
- осознанного выбора подхода в зависимости от клинической задачи,
- критической оценки этических аспектов автоматического анализа медицинских текстов.

Работа выполняется в среде Python с использованием библиотек `nltk`, `pymorphy3`, `scikit-learn` (и, по желанию, `gensim` для эмбеддингов). Все задания сопровождаются пояснениями и примерами.

---

## Подготовка рабочей среды

Перед началом работы установите необходимые библиотеки (в терминале или командной строке):

```python
!pip install nltk pymorphy3 scikit-learn pandas matplotlib
# Для работы с эмбеддингами (дополнительно):
!pip install gensim
```

Затем в Python-скрипте или Jupyter Notebook импортируйте модули и скачайте данные для NLTK:

```python
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')   # если требуется
```

---

## Часть 1. Теоретические вопросы (для самопроверки)

Перед выполнением практических заданий ответьте письменно на следующие вопросы (кратко, но содержательно). Это поможет убедиться, что вы понимаете ключевые концепции.

1. Почему медицинские тексты (жалобы пациентов, записи в карте) считаются «шумными»? Приведите три примера такого шума.
2. Что такое токенизация и для чего она нужна при анализе клинических записей?
3. Почему при удалении стоп‑слов в медицинских текстах важно **не** удалять частицу «не» и другие отрицания («нет», «без», «отсутствует»)? Приведите пример, где это критично для диагностики.
4. В чём разница между стеммингом и лемматизацией? Какой метод больше подходит для русского языка и почему?
5. Для каких задач в стоматологии подходит **One‑Hot Encoding**, а для каких — нет? Приведите примеры.
6. Что такое «мешок слов» (Bag of Words)? Опишите его сильные и слабые стороны при анализе жалоб пациентов.
7. В чём преимущество метода TF‑IDF перед простым подсчётом частот? Приведите пример слова, которое получит высокий вес в тексте о послеоперационном осложнении (например, «альвеолит»).
8. Что такое векторные эмбеддинги слов и как они помогают компьютеру «понимать» смысл медицинских терминов?
9. Назовите три особенности русского языка, усложняющих его обработку методами NLP, с примерами из стоматологической лексики.
10. Почему при анализе медицинских текстов на русском языке не рекомендуется переводить их на английский и использовать англоязычные модели? Приведите пример с описанием боли.

---

## Часть 2. Практические задания на Python

Все задания выполняйте в одной Jupyter Notebook или в отдельном Python‑скрипте. Код должен быть прокомментирован. В конце каждого задания приводите краткий вывод.

---

### Задание 1. Очистка текста жалобы пациента

**Описание.** Возьмите следующий текст — реальное сообщение пациента в онлайн-чат стоматологической клиники (с опечатками, сленгом и эмоциональной лексикой):

```
«здраствуйте, у миня зуб балел вчера, а сиводня апухла щека и баюсь што флюс. чё делать?»
```

Выполните последовательно все этапы препроцессинга и выведите результаты на каждом шаге.

**Требуется:**

1. **Токенизация** — разбить текст на отдельные токены (слова и знаки препинания). Используйте `nltk.word_tokenize(text, language='russian')`.
2. **Удаление стоп‑слов** — удалить стандартные стоп‑слова русского языка (из `nltk.corpus.stopwords.words('russian')`), но **обязательно сохранить** отрицания («не», «нет», «без»), если они встретятся. Выведите список оставшихся слов.
3. **Нормализация** — вручную или с помощью регулярных выражений исправить очевидные опечатки и сленг (например, «здраствуйте» → «здравствуйте», «миня» → «меня», «балел» → «болел», «сиводня» → «сегодня», «апухла» → «опухла», «баюсь» → «боюсь», «што» → «что», «флюс» — оставить, это медицинский сленг, но при желании можно заменить на «периостит», «чё» → «что»). Можно также привести все слова к нижнему регистру.
4. **Лемматизация** — привести все слова к нормальной (словарной) форме с помощью `pymorphy3.MorphAnalyzer()`. Выведите итоговый список лемм.

**Результат:** вы должны получить чистый список смысловых единиц, например: `['здравствовать', 'я', 'зуб', 'болеть', 'вчера', 'сегодня', 'опухнуть', 'щека', 'бояться', 'что', 'флюс', 'что', 'делать']`.

**Вопрос для размышления:** Какие ключевые клинические маркеры вы видите в очищенном тексте? Опишите их (острая боль, отёк, страх, подозрение на флюс/периостит). Какие слова являются семантически важными для постановки предварительного диагноза?

```python
# Ваш код решения задачи:
```

---

### Задание 2. Векторизация методом Bag of Words и TF‑IDF

**Описание.** Даны три короткие записи из амбулаторных карт (после лемматизации):

1. *«пациент жаловаться боль зуб накусывание»*
2. *«боль проходить но оставаться отёк десна»*
3. *«пациент чувствовать пульсирующий боль ночь и отёк щека»*

**2.1. Постройте матрицу «мешок слов» (Bag of Words).**

- Используйте `sklearn.feature_extraction.text.CountVectorizer`.
- Выведите общий словарь (названия признаков) и саму матрицу частот (в виде массива).
- Объясните, какие слова встречаются чаще всего и почему.

**2.2. Вычислите TF‑IDF‑веса для этих же текстов.**

- Используйте `sklearn.feature_extraction.text.TfidfVectorizer`.
- Выведите матрицу TF‑IDF.
- Сравните значения для частотных слов (например, «боль», «пациент») и для редких («накусывание», «пульсирующий», «ночь»). Какой метод лучше выделяет клинически значимые слова?

**2.3. Визуализация (по желанию).**

- Постройте тепловую карту (heatmap) матрицы частот или TF‑IDF с помощью `seaborn` или `matplotlib`. Подпишите строки (тексты) и столбцы (слова).

```python
# Ваш код решения задачи:
```

---

### Задание 3. Работа с векторными эмбеддингами (введение)

**Описание.** В этом задании мы попробуем использовать предобученные векторные представления слов (Word2Vec) для измерения семантической близости между медицинскими терминами, важными в стоматологии.

**Инструкция:**

1. Загрузите предобученную модель `fasttext-wiki-news-subwords-300` через `gensim.downloader` (около 500 МБ, может занять время). Если загрузка затруднена, можно использовать альтернативный подход (например, модель `spacy` с русской моделью `ru_core_news_sm`), но для единообразия предложим первый вариант.

```python
import gensim.downloader as api
model = api.load("fasttext-wiki-news-subwords-300")  # может занять время
```

2. Получите векторы для следующих слов (стоматологических терминов и состояний):
   - `пульпит`
   - `периодонтит`
   - `кариес`
   - `гингивит`
   - `боль`
   - `отёк`
   - `анестезия`
   - `страх`
   - `лечение`
   - `удаление`

3. Вычислите косинусное расстояние между каждой парой слов из этого списка (можно построить матрицу расстояний). С помощью комментариев объясните, какие пары оказались семантически близкими, а какие — далёкими. Соответствует ли это клиническим представлениям? Например, «пульпит» и «периодонтит» должны быть ближе, чем «пульпит» и «отбеливание» (если включить такое слово).

4. *Дополнительное задание (для углублённого понимания).* Выберите слово «осложнение» и найдите 5 самых похожих по смыслу слов в модели. Сравните результат с тем, что вы ожидали как врач (например, «альвеолит», «периимплантит», «кровотечение»).

**Результат:** вы должны представить код и краткий текстовый анализ полученных семантических отношений.

```python
# Ваш код решения задачи:
```

---

### Задание 4. Этический анализ системы NLP в стоматологии

**Описание.** Представьте, что вы — руководитель стоматологической клиники. Вам предлагают внедрить систему, которая автоматически анализирует сообщения пациентов из онлайн-чата и отзывы после лечения с помощью описанных методов NLP (от очистки до эмбеддингов) и выдаёт предупреждения о возможных осложнениях (например, подозрение на альвеолит, периимплантит, неадекватную реакцию на анестезию).

Напишите эссе (объёмом 1–2 страницы) на тему **«Этические вызовы при автоматическом анализе медицинских текстов»**. В эссе обязательно осветите следующие аспекты:

1. **Информированное согласие** — что должно быть разъяснено пациенту перед началом использования системы? Какие пункты обязательно включить в форму согласия?
2. **Конфиденциальность и безопасность данных** — как должны храниться и обрабатываться тексты пациентов? Кто имеет доступ к результатам анализа? Как обеспечивается анонимизация?
3. **Ответственность за ошибки** — кто несёт ответственность, если алгоритм пропустит признаки серьёзного осложнения или, наоборот, ложно сработает и вызовет ненужную тревогу у пациента?
4. **Прозрачность и интерпретируемость** — должен ли пациент знать, как именно работает алгоритм? Как врач может объяснить пациенту, что анализ проводится автоматически?

Постарайтесь привести аргументированные предложения по минимизации каждого из рисков. Ссылайтесь на материалы лекции (раздел 9, 10 и этические замечания).

```text
# Ваш текст эссе:
```

---

## Часть 3. Комплексное задание (повышенной сложности)

**Классификация текстов жалоб на наличие острого воспалительного процесса (по желанию).**

Если вы уже знакомы с основами машинного обучения, попробуйте реализовать простой классификатор, который по тексту определяет, относится ли он к острому воспалительному процессу (например, периостит, альвеолит) или нет.

**Инструкция:**

- Используйте небольшой размеченный датасет (например, из открытых источников, либо создайте синтетическую выборку из 20–30 коротких жалоб с метками: 1 — острое воспаление (есть слова «опухла», «гной», «температура», «дёргает»), 0 — плановая ситуация (например, «хочу записаться на осмотр», «после лечения всё хорошо»)).
- Примените TF‑IDF для извлечения признаков.
- Обучите классификатор (логистическая регрессия, наивный Байес или метод опорных векторов) с помощью `sklearn`.
- Оцените точность (accuracy) на тестовой выборке (или используйте кросс‑валидацию).
- Опишите, какие слова (признаки) оказались наиболее информативными для классификации.

Это задание не является обязательным, но даст вам более глубокое понимание всего пайплайна NLP в клинической задаче.

---

## Критерии оценки

- **Теоретические вопросы** — 20% (полнота и правильность ответов).
- **Задание 1 (очистка)** — 25% (правильность выполнения всех этапов, комментарии, вывод).
- **Задание 2 (BoW и TF‑IDF)** — 25% (корректность кода, интерпретация результатов).
- **Задание 3 (эмбеддинги)** — 15% (код работает, анализ расстояний обоснован).
- **Задание 4 (этика)** — 15% (глубина анализа, аргументированность, соответствие теме).
- Дополнительное задание (классификация) — оценивается отдельно как бонус.

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb` или Python `.py`) со всеми заданиями.
- Код должен быть чистым, снабжённым комментариями на русском языке.
- Ответы на теоретические вопросы и эссе по этике приложите в виде текстовых ячеек (в Notebook) или отдельного документа (если сдаёте скрипт, то отдельный файл `.txt` или `.pdf`).
- Убедитесь, что все библиотеки импортированы и код выполняется без ошибок (при необходимости укажите версии).

---

## Срок выполнения

Работа рассчитана на **2 недели**. Рекомендуется выполнять задания последовательно, после каждой лекции по NLP.

---

## Заключение

Данная практическая работа охватывает полный цикл обработки медицинского текста — от «сырого» языка пациента и врача до числовых векторов и этических размышлений. Вы не только освоите инструменты, но и научитесь критически оценивать их применение в клинической практике. Успехов!